# NeuroArch SNN Training Demo
Reproduces accuracy curves from paper Fig. 3

In [ ]:
import sys; sys.path.insert(0,'../snn')
import torch, numpy as np
from model import LIFComfortClassifier
from rate_encoder import rate_encode
print('Model parameters:', sum(p.numel() for p in LIFComfortClassifier().parameters()))
print('Expected: 3,104 synapses (Table 2)')

In [ ]:
# Load pre-trained weights and evaluate
import os
model = LIFComfortClassifier(T=100)
weights_path = '../snn/weights/neuroarch_office.pt'
if os.path.exists(weights_path):
    model.load_state_dict(torch.load(weights_path, map_location='cpu'))
    print('Loaded pre-trained weights')
else:
    print('No pre-trained weights found; see snn/train.py to train from scratch')

In [ ]:
# Simulate spike raster for a Warm comfort event (paper Fig. 10)
import matplotlib.pyplot as plt
np.random.seed(42)
# Warm event: high temp (0.85), high RH (0.75), low CO2 (0.2)
s = torch.zeros(14)
s[0]=0.85; s[2]=0.75; s[8]=0.20  # T, RH, CO2
spikes = rate_encode(s.unsqueeze(0), T=100).squeeze(1).numpy()
plt.figure(figsize=(10,3))
for ch in range(14):
    t_spikes = np.where(spikes[:,ch])[0]
    plt.vlines(t_spikes, ch+0.1, ch+0.9, color='steelblue', alpha=0.7)
plt.xlabel('Timestep (ms)'); plt.ylabel('Input channel'); plt.title('Spike Raster – Warm Event (T=0.85, RH=0.75)')
plt.tight_layout(); plt.savefig('spike_raster_demo.png', dpi=150); plt.show()
print('Channel 0 (Temperature) spike count:', spikes[:,0].sum())